# Lab 04 — Thành viên 1: Parser & Discovery

**Nhiệm vụ:**
1. Clone repo, enumerate file `.py`, loại test/setup/auto-gen (Task 1)
2. Xây Parser Service: dùng `ast` stdlib, trích AST/CFG/DFG/Call edges (Task 2)
3. Đảm bảo Stable ID cho mọi element (đảm bảo idempotency cho Task 6)
4. Định nghĩa schema message để Thành viên 2 dùng ngay cho Kafka topic

**Thư viện:** `ast` (stdlib Python 3.10+) — không cần cài thêm gói


---
## 0. Setup — Import & Paths

In [1]:
import sys
import os
import json

# Thêm thư mục src vào Python path
REPO_ROOT     = os.path.abspath('../lerobot')
PROJECT_ROOT  = os.path.abspath('..')
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

print(f'Project root : {PROJECT_ROOT}')
print(f'Lerobot repo : {REPO_ROOT}')
print(f'Python version: {sys.version}')

Project root : d:\HuynhHan\Bigdata\bigdata-lab04-pipeline
Lerobot repo : d:\HuynhHan\Bigdata\bigdata-lab04-pipeline\lerobot
Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


---
## Task 1: Repository Cloning & File Discovery

**Repo được chọn:** [huggingface/lerobot](https://github.com/huggingface/lerobot)

```bash
# Lệnh clone (đã thực hiện):
git clone --depth=1 https://github.com/huggingface/lerobot.git lerobot
```

### Tiêu chí lọc file:
| Loại | Ví dụ | Lý do loại bỏ |
|------|-------|----------------|
| Test files | `tests/` | Không phải code core |
| Setup files | `setup.py`, `conftest.py` | Packaging/test config |
| Auto-generated | `*_pb2.py`, `*_generated.py` | Code tạo tự động |
| Build artifacts | `build/`, `dist/`, `__pycache__/` | Không phải source |
| Examples | `examples/` | Demo, không phải production code |

In [2]:
from discovery import discover_python_files, EXCLUDE_DIRS, EXCLUDE_FILES, EXCLUDE_SUFFIXES

print('=== Cấu hình lọc ===')
print(f'Thư mục bị loại: {sorted(EXCLUDE_DIRS)}')
print(f'File bị loại   : {sorted(EXCLUDE_FILES)}')
print(f'Suffix bị loại : {EXCLUDE_SUFFIXES}')

=== Cấu hình lọc ===
Thư mục bị loại: ['.eggs', '.git', '.github', '.mypy_cache', '.pytest_cache', '.tox', '.venv', '__pycache__', 'build', 'dist', 'env', 'examples', 'htmlcov', 'node_modules', 'tests', 'venv']
File bị loại   : ['conftest.py', 'manage.py', 'setup.cfg', 'setup.py']
Suffix bị loại : ('_pb2.py', '_pb2_grpc.py', '_generated.py', '_gen.py')


In [3]:
# Khám phá toàn bộ file Python trong repo lerobot
py_files = discover_python_files(REPO_ROOT)

print(f'\Tổng số file Python cốt lõi phát hiện được: {len(py_files)}')
print('\n10 file đầu tiên:')
for f in py_files[:10]:
    print(f"  [{f['file_size_bytes']:>8} bytes] {f['relative_path']}")

\Tổng số file Python cốt lõi phát hiện được: 491

10 file đầu tiên:
  [    8838 bytes] scripts\ci\extract_task_descriptions.py
  [    5116 bytes] scripts\ci\parse_eval_metrics.py
  [    2016 bytes] src\lerobot\__init__.py
  [     857 bytes] src\lerobot\__version__.py
  [    1814 bytes] src\lerobot\types.py
  [     649 bytes] src\lerobot\annotations\__init__.py
  [    1465 bytes] src\lerobot\annotations\steerable_pipeline\__init__.py
  [    8893 bytes] src\lerobot\annotations\steerable_pipeline\config.py
  [   10781 bytes] src\lerobot\annotations\steerable_pipeline\executor.py
  [   21642 bytes] src\lerobot\annotations\steerable_pipeline\frames.py


<>:4: SyntaxWarning: invalid escape sequence '\T'
<>:4: SyntaxWarning: invalid escape sequence '\T'
C:\Users\admin\AppData\Local\Temp\ipykernel_12168\2364593198.py:4: SyntaxWarning: invalid escape sequence '\T'
  print(f'\Tổng số file Python cốt lõi phát hiện được: {len(py_files)}')


In [4]:
import collections

# Thống kê phân bổ file theo thư mục cấp 1
dir_counts = collections.Counter()
for f in py_files:
    parts = f['relative_path'].replace('\\', '/').split('/')
    dir_counts[parts[0]] += 1

print('=== Phân bổ file theo thư mục top-level ===')
for d, c in dir_counts.most_common():
    bar = '█' * (c // 5)
    print(f'  {d:<30} {c:>4} files  {bar}')

total_size = sum(f['file_size_bytes'] for f in py_files)
print(f'\nTổng kích thước: {total_size / 1024:.1f} KB')

=== Phân bổ file theo thư mục top-level ===
  src                             489 files  █████████████████████████████████████████████████████████████████████████████████████████████████
  scripts                           2 files  

Tổng kích thước: 5670.7 KB


In [5]:
# Lưu danh sách ra JSON để Parser Service dùng
output_path = os.path.join(PROJECT_ROOT, 'discovered_files.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(py_files, f, indent=2, ensure_ascii=False)

print(f'Đã lưu danh sách {len(py_files)} file vào: {output_path}')

# Kiểm tra file first entry
print('\nSample entry trong discovered_files.json:')
print(json.dumps(py_files[0], indent=2))

Đã lưu danh sách 491 file vào: d:\HuynhHan\Bigdata\bigdata-lab04-pipeline\discovered_files.json

Sample entry trong discovered_files.json:
{
  "relative_path": "scripts\\ci\\extract_task_descriptions.py",
  "absolute_path": "d:\\HuynhHan\\Bigdata\\bigdata-lab04-pipeline\\lerobot\\scripts\\ci\\extract_task_descriptions.py",
  "file_size_bytes": 8838,
  "file_hash": "c359c4a6f0ab80a1df334513305656dbae4b87e7681b24ade59354b7a0640043"
}


---
## Kafka Topic Schema (Giao diện cho Thành viên 2)

Tất cả schema và tên topic được định nghĩa trong `src/schemas.py`.
Thành viên 2 chỉ cần import và dùng — **không cần tự định nghĩa**.

In [6]:
from schemas import (
    TOPIC_NODES, TOPIC_EDGES, TOPIC_METADATA, TOPIC_ERRORS,
    make_node_event, make_edge_event, make_metadata_event, make_error_event
)

print('=== Kafka Topic Layout ===')
print(f'  cpg.nodes      → {TOPIC_NODES}')
print(f'  cpg.edges      → {TOPIC_EDGES}')
print(f'  cpg.metadata   → {TOPIC_METADATA}')
print(f'  cpg.errors     → {TOPIC_ERRORS}')

print('\n=== Sample Node Event Schema ===')
sample_node = make_node_event(
    node_id='node_abc123def456',
    file_path='src/lerobot/__init__.py',
    label='FunctionDef',
    node_type='FunctionDef',
    line_number=10,
    col_offset=0,
    end_lineno=25,
    end_col_offset=4,
    name='my_function',
    code_snippet='def my_function(arg1, arg2):',
    scope='MyClass',
)
print(json.dumps(sample_node, indent=2))

=== Kafka Topic Layout ===
  cpg.nodes      → cpg.nodes
  cpg.edges      → cpg.edges
  cpg.metadata   → cpg.metadata
  cpg.errors     → cpg.errors

=== Sample Node Event Schema ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-19T16:20:16.655947Z",
  "topic": "cpg.nodes",
  "file_path": "src/lerobot/__init__.py",
  "node_id": "node_abc123def456",
  "label": "FunctionDef",
  "properties": {
    "type": "FunctionDef",
    "line_number": 10,
    "col_offset": 0,
    "end_lineno": 25,
    "end_col_offset": 4,
    "name": "my_function",
    "code_snippet": "def my_function(arg1, arg2):",
    "scope": "MyClass"
  }
}


In [7]:
print('=== Sample Edge Event Schema ===')
sample_edge = make_edge_event(
    edge_id='edge_789xyz',
    file_path='src/lerobot/__init__.py',
    edge_type='CFG_BRANCH_TRUE',
    source_id='node_if_condition',
    target_id='node_then_body',
    properties={'branch': 'true_branch'},
)
print(json.dumps(sample_edge, indent=2))

print('\n=== Sample Error Event Schema ===')
sample_err = make_error_event(
    file_path='src/lerobot/broken.py',
    error_type='SyntaxError',
    error_message='invalid syntax',
    line_number=42,
    col_offset=7,
)
print(json.dumps(sample_err, indent=2))

=== Sample Edge Event Schema ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-19T16:20:16.663029Z",
  "topic": "cpg.edges",
  "file_path": "src/lerobot/__init__.py",
  "edge_id": "edge_789xyz",
  "type": "CFG_BRANCH_TRUE",
  "source_id": "node_if_condition",
  "target_id": "node_then_body",
  "properties": {
    "branch": "true_branch"
  }
}

=== Sample Error Event Schema ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-19T16:20:16.663211Z",
  "topic": "cpg.errors",
  "file_path": "src/lerobot/broken.py",
  "error_type": "SyntaxError",
  "error_message": "invalid syntax",
  "line_number": 42,
  "col_offset": 7
}


---
## Task 2: Incremental CPG Parser Service

### Thư viện được chọn: `ast` (Python Standard Library)

| Tiêu chí | `ast` stdlib | `tree-sitter` | `Joern` |
|----------|:-----------:|:-------------:|:-------:|
| Không cần cài | ✅ | ❌ | ❌ |
| Python-native | ✅ | Partial | ❌ |
| AST nodes | ✅ | ✅ | ✅ |
| CFG edges | ⚠️ Manual | ✅ | ✅ |
| DFG edges | ⚠️ Manual | ⚠️ Partial | ✅ |
| Call edges | ✅ Manual | ✅ | ✅ |

**Lý do chọn `ast`:** Phù hợp nhất cho môi trường học tập, không cần dependency ngoài, đủ để demonstrate 4 loại edge theo yêu cầu.

### Stable ID Strategy (quan trọng cho Task 6)

```
node_id = SHA-256( file_path + "::" + node_type + "@L" + lineno + "C" + col_offset )[:16]
```

**Tại sao KHÔNG dùng `id(ast_node)`:**
- `id()` là địa chỉ bộ nhớ → thay đổi mỗi lần parse → duplicate nodes trong Neo4j
- SHA-256 của vị trí source code → deterministic → same result every run

In [8]:
from parser_service import CPGParser

# Chọn 1 file để demo
demo_file = py_files[2]['absolute_path']  # lerobot/src/lerobot/types.py
print(f'Demo file: {py_files[2]["relative_path"]}')
print(f'Size: {py_files[2]["file_size_bytes"]} bytes')

Demo file: src\lerobot\__init__.py
Size: 2016 bytes


In [9]:
# Run parser
parser = CPGParser(absolute_path=demo_file, repo_root=REPO_ROOT)
nodes, edges, metadata, error_event = parser.parse()

if error_event:
    print(f'Parse error: {error_event["error_type"]} — {error_event["error_message"]}')
else:
    print('Parse thành công!')
    print(f'\nKết quả parse:')
    print(f'  Tổng AST nodes : {metadata["total_nodes"]}')
    print(f'  AST edges      : {metadata["total_edges"]["ast"]}')
    print(f'  CFG edges      : {metadata["total_edges"]["cfg"]}')
    print(f'  DFG edges      : {metadata["total_edges"]["dfg"]}')
    print(f'  CALL edges     : {metadata["total_edges"]["call"]}')
    print(f'  File hash      : {metadata["file_hash"]}')
    print(f'  Parse time     : {metadata["parse_duration_ms"]} ms')

Parse thành công!

Kết quả parse:
  Tổng AST nodes : 58
  AST edges      : 57
  CFG edges      : 3
  DFG edges      : 0
  CALL edges     : 0
  File hash      : 5c2fb7720e0a0a26bc13a15ac2089f62d8f7ba9f531923d5a550dd95d8b8f1a0
  Parse time     : 2.45 ms


In [10]:
# Xem chi tiết các loại node
import collections

node_types = collections.Counter(n['properties']['type'] for n in nodes)
print('=== Phân bổ loại AST Node (Top 15) ===')
for ntype, cnt in node_types.most_common(15):
    bar = '█' * cnt
    print(f'  {ntype:<25} {cnt:>4}  {bar}')

=== Phân bổ loại AST Node (Top 15) ===
  Constant                    21  █████████████████████
  Load                        13  █████████████
  Name                         6  ██████
  List                         6  ██████
  Subscript                    2  ██
  Store                        2  ██
  Module                       1  █
  Expr                         1  █
  ImportFrom                   1  █
  AnnAssign                    1  █
  Assign                       1  █
  alias                        1  █
  Dict                         1  █
  Tuple                        1  █


In [11]:
# Xem sample node event
print('=== Sample Node Event (sẽ produce lên cpg.nodes) ===')
# Tìm FunctionDef node để demo
func_nodes = [n for n in nodes if n['properties']['type'] == 'FunctionDef']
if func_nodes:
    print(json.dumps(func_nodes[0], indent=2))
else:
    print(json.dumps(nodes[0], indent=2))

=== Sample Node Event (sẽ produce lên cpg.nodes) ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-19T16:20:16.681988Z",
  "topic": "cpg.nodes",
  "file_path": "src/lerobot/__init__.py",
  "node_id": "node_648ef74f6c9c0a3a",
  "label": "AST_Node",
  "properties": {
    "type": "Module",
    "line_number": null,
    "col_offset": null,
    "end_lineno": null,
    "end_col_offset": null,
    "name": null,
    "code_snippet": null,
    "scope": null
  }
}


In [12]:
# Xem phân bổ loại edge
edge_types = collections.Counter(e['type'] for e in edges)
print('=== Phân bổ loại Edge ===')
for etype, cnt in edge_types.most_common():
    bar = '█' * (cnt // 2 + 1)
    print(f'  {etype:<22} {cnt:>4}  {bar}')

=== Phân bổ loại Edge ===
  AST_CHILD                57  █████████████████████████████
  CFG_NEXT                  3  ██


In [13]:
# Sample CFG edge
cfg_edges = [e for e in edges if e['type'].startswith('CFG')]
if cfg_edges:
    print('=== Sample CFG Edge ===')
    print(json.dumps(cfg_edges[0], indent=2))
else:
    print('Không có CFG edges trong file này (file không có control flow)')

=== Sample CFG Edge ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-19T16:20:16.683012Z",
  "topic": "cpg.edges",
  "file_path": "src/lerobot/__init__.py",
  "edge_id": "edge_0940af51f37efdd7",
  "type": "CFG_NEXT",
  "source_id": "node_3ceb920a698239ca",
  "target_id": "node_e58c45e1e96d4ce6",
  "properties": {
    "sequence": 0
  }
}


In [14]:
# Sample DFG edge
dfg_edges = [e for e in edges if e['type'] == 'DFG_USE']
if dfg_edges:
    print('=== Sample DFG Edge ===')
    print(json.dumps(dfg_edges[0], indent=2))
    print(f'Biến được track: {dfg_edges[0]["properties"]["variable_name"]}')
else:
    print('Không có DFG edges trong file này')

Không có DFG edges trong file này


In [15]:
# Sample CALL edge
call_edges = [e for e in edges if 'CALL' in e['type']]
if call_edges:
    print('=== Sample CALL Edge ===')
    print(json.dumps(call_edges[0], indent=2))
else:
    print('Không có CALL edges trong file này')

Không có CALL edges trong file này


In [16]:
# Metadata event (sẽ produce lên cpg.metadata → Spark → MongoDB)
print('=== Metadata Event (cho cpg.metadata topic) ===')
print(json.dumps(metadata, indent=2))

=== Metadata Event (cho cpg.metadata topic) ===
{
  "schema_version": "1.0",
  "event_time": "2026-07-19T16:20:16.683668Z",
  "topic": "cpg.metadata",
  "file_path": "src/lerobot/__init__.py",
  "file_size_bytes": 2016,
  "file_hash": "5c2fb7720e0a0a26bc13a15ac2089f62d8f7ba9f531923d5a550dd95d8b8f1a0",
  "total_nodes": 58,
  "total_edges": {
    "ast": 57,
    "cfg": 3,
    "dfg": 0,
    "call": 0
  },
  "parser_version": "ast-stdlib-3.x",
  "parse_duration_ms": 2.45
}


---
## Kiểm chứng Stable ID & Idempotency (chuẩn bị cho Task 6)

**Yêu cầu:** Parse cùng 1 file nhiều lần phải sinh ra **cùng bộ node_id và edge_id**.
Điều này đảm bảo Neo4j dùng `MERGE` sẽ không tạo duplicate nodes/edges.

In [17]:
print('=== Kiểm tra Idempotency: parse 3 lần cùng file ===')

all_node_ids = []
all_edge_ids = []

for i in range(3):
    p = CPGParser(absolute_path=demo_file, repo_root=REPO_ROOT)
    n, e, m, err = p.parse()
    node_ids = sorted([x['node_id'] for x in n])
    edge_ids = sorted([x['edge_id'] for x in e])
    all_node_ids.append(node_ids)
    all_edge_ids.append(edge_ids)
    print(f'  Run {i+1}: {len(n)} nodes, {len(e)} edges')

nodes_stable = all(all_node_ids[0] == r for r in all_node_ids)
edges_stable = all(all_edge_ids[0] == r for r in all_edge_ids)

print()
print(f'Node IDs stable across 3 runs: {"YES" if nodes_stable else "NO"}')
print(f'Edge IDs stable across 3 runs: {"YES" if edges_stable else "NO"}')
print()
if nodes_stable and edges_stable:
    print('IDEMPOTENCY: PASS — Neo4j MERGE sẽ không tạo duplicate!')
else:
    print('IDEMPOTENCY: FAIL')

=== Kiểm tra Idempotency: parse 3 lần cùng file ===
  Run 1: 58 nodes, 60 edges
  Run 2: 58 nodes, 60 edges
  Run 3: 58 nodes, 60 edges

Node IDs stable across 3 runs: YES
Edge IDs stable across 3 runs: YES

IDEMPOTENCY: PASS — Neo4j MERGE sẽ không tạo duplicate!


In [18]:
# Demo: xem ID thực tế
import hashlib

print('=== Ví dụ về Stable ID vs. Memory-address ID ===')
import ast

test_code = '''
def greet(name):
    message = f"Hello, {name}!"
    return message
'''

ids_by_memory = []
ids_by_stable = []

for _ in range(3):
    tree = ast.parse(test_code)
    func_node = next(n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef))
    
    # Cách cũ (BUG): dùng id() — memory address
    ids_by_memory.append(id(func_node))
    
    # Cách mới (CORRECT): SHA-256 của vị trí
    raw = f'test.py::FunctionDef@L{func_node.lineno}C{func_node.col_offset}'
    ids_by_stable.append('node_' + hashlib.sha256(raw.encode()).hexdigest()[:16])

print(f'Memory-based IDs (BUG):  {ids_by_memory}')
print(f'All different?           {len(set(ids_by_memory)) > 1}')
print()
print(f'SHA256-based IDs (FIXED): {ids_by_stable}')
print(f'All same?                 {len(set(ids_by_stable)) == 1}')

=== Ví dụ về Stable ID vs. Memory-address ID ===
Memory-based IDs (BUG):  [1625357582096, 1625357577808, 1625357576848]
All different?           True

SHA256-based IDs (FIXED): ['node_b53008f3b0d4d05b', 'node_b53008f3b0d4d05b', 'node_b53008f3b0d4d05b']
All same?                 True


---
## Pipeline Demo: Xử lý 5 File Đầu Tiên

Demo toàn bộ pipeline incremental: mỗi file được parse độc lập, kết quả được in ra như thể đang produce lên Kafka.

In [19]:
print('=== Demo: Xử lý 5 file đầu tiên theo kiểu incremental ===')
print(f'{"File":<55} {"Nodes":>6} {"AST":>5} {"CFG":>5} {"DFG":>5} {"CALL":>5} {"ms":>6}')
print('-' * 95)

# Lấy file có content phong phú (size > 1KB)
interesting_files = [f for f in py_files if f['file_size_bytes'] > 1000][:5]

summary = []
for file_info in interesting_files:
    parser = CPGParser(absolute_path=file_info['absolute_path'], repo_root=REPO_ROOT)
    nodes, edges, meta, err = parser.parse()
    
    if err:
        print(f'{file_info["relative_path"]:<55} ERROR: {err["error_type"]}')
        continue
    
    e = meta['total_edges']
    print(f'{file_info["relative_path"]:<55} {meta["total_nodes"]:>6} {e["ast"]:>5} {e["cfg"]:>5} {e["dfg"]:>5} {e["call"]:>5} {meta["parse_duration_ms"]:>6.1f}')
    summary.append(meta)

print('-' * 95)
if summary:
    total_nodes = sum(m['total_nodes'] for m in summary)
    total_edges = sum(sum(m['total_edges'].values()) for m in summary)
    print(f'{"TOTAL (5 files)":<55} {total_nodes:>6} {total_edges:>22}')

=== Demo: Xử lý 5 file đầu tiên theo kiểu incremental ===
File                                                     Nodes   AST   CFG   DFG  CALL     ms
-----------------------------------------------------------------------------------------------
scripts\ci\extract_task_descriptions.py                   1003  1002    81   124    65   30.9
scripts\ci\parse_eval_metrics.py                           607   606    56    51    38   25.1
src\lerobot\__init__.py                                     58    57     3     0     0    3.2
src\lerobot\types.py                                       227   226    18     4     1   10.0
src\lerobot\annotations\steerable_pipeline\__init__.py      19    18     4     0     0    1.4
-----------------------------------------------------------------------------------------------
TOTAL (5 files)                                           1914                   2354


---
## Tổng kết Thành viên 1

| Nhiệm vụ | Ghi chú |
|----------|--------|
| Clone repo & enumerate `.py` files | 493 files, lưu vào `discovered_files.json` |
| Loại test/setup/auto-gen | 5 loại thư mục + 2 loại file + suffix patterns |
| Parser Service (AST) | `ast` stdlib |
| CFG edges | sequential + branching + try/except |
| DFG edges | def→use analysis |
| CALL edges | internal + external calls |
| Stable ID (idempotency) | SHA-256 deterministic, PASS qua 3 lần parse |
| Schema message cho Thành viên 2 | `schemas.py` với 4 schemas + 4 topic constants |

### Handoff cho Thành viên 2:
```python
from schemas import TOPIC_NODES, TOPIC_EDGES, TOPIC_METADATA, TOPIC_ERRORS
from parser_service import CPGParser

for file_info in py_files:
    parser = CPGParser(file_info['absolute_path'], REPO_ROOT)
    nodes, edges, metadata, error = parser.parse()
    
    # Thành viên 2 produce events lên Kafka:
    for event in nodes:    producer.send(TOPIC_NODES,    value=event)
    for event in edges:    producer.send(TOPIC_EDGES,    value=event)
    producer.send(TOPIC_METADATA, value=metadata)
    if error: producer.send(TOPIC_ERRORS, value=error)
```